# T7 - environmental seasonality 
An individual’s dose through long-cycle transmission in the model is attenuated or amplified by a mechanism for seasonality. Seasonality can represent an attenuation of the likely constant shedding of individuals into the long-cycle vehicle; or it could be a temperature-dependent growth in the level of CFUs in the environmental pool. 

In [ ]:
import starsim as ss
import typhoidsim as ty

# Define high-level simulation parameters
pars = dict(
    start    = 2000,  # Starting year
    n_years  = 10.0,   # Duration of the simulation in years
    dt       = 1.0/365.0,     # Timestep of 1 day, expressed in years
    verbose  = 0,             # Do not print details of the run
)

# The population
ppl = ss.People(10_000)

# The disease, with reduced protection per infection `tppi` 
typhoid = ty.Typhoid(pars={'tppi':0.1})


# The environment (long cycle CCVT)
environment = ty.EnvironmentalPool()

# Modulates the level of CFUs int the environment
seasonal_pattern = ty.Pattern("baseline_cfu + amp_cfu * cos((2*pi/period)*var)",
                              pars={'baseline_cfu': 4_000_000,
                                    'amp_cfu': 1_000_000,
                                    'period': 0.5,  # in years
                                    'pi': 3.141592653589793})

# This intervention adds to the current CFU level in the environment 
bacterial_growth = ty.environmental_seasonality(seasonal_pattern=seasonal_pattern)

sim = ss.Sim(
    pars=pars,
    diseases=typhoid,
    demographics=environment,
    interventions=bacterial_growth,
    )

sim.run()
sim.plot(key=('environmentalpool_cfu', 'environmentalpool_temp'))

### Attenuate the level of exposure by reducing the shedding rate into the environment

In [ ]:
import starsim as ss
import typhoidsim as ty

# Define high-level simulation parameters
pars = dict(
    start    = 2000,  # Starting year
    n_years  = 10.0,   # Duration of the simulation in years
    dt       = 1.0/365.0,     # Timestep of 1 day, expressed in years
    verbose  = 0,             # Do not print details of the run
)

# The population
ppl = ss.People(10_000)

# The disease, with reduced protection per infection `tppi` 
typhoid = ty.Typhoid(pars={'tppi':0.1})


# The environment (long cycle CCVT)
environment = ty.EnvironmentalPool()

# Modulates the level of CFUs int the environment
sanitation_efficacy = ty.Pattern("baseline_eff + amp_eff * cos((2*pi/period)*var)",
                              pars={'baseline_eff': 0.5,  # baseline efficacy in reducing shedding
                                    'amp_eff': 0.25,      # oscillation of efficacy around the mean
                                    'period': 1.0,        # in years
                                    'pi': 3.141592653589793})

sanitation_efficacy = ty.Pattern("var % period < width",
                              pars={'period': 1.0,        # in years
                                    'width' : 0.5}        # in years
                                )

sanitation = ty.shedding_reduction(efficacy=sanitation_efficacy, start=2005)

sim = ss.Sim(
    pars=pars,
    diseases=typhoid,
    demographics=environment,
    interventions=sanitation
    )

sim.run()
sim.plot(key=('environmentalpool_cfu', 'environmentalpool_temp'))

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(sim.interventions.shedding_reduction.results.efficacy, label='Efficacy profile')
plt.plot(sim.interventions.shedding_reduction.results.effective_value, label='Effective shedding rate')


plt.axvline(x=2005, color='k', ls='--')
plt.title('Seasonal shedding reduction')
plt.legend()
plt.show()